# Entrenamiento de modelo IA a partir de Resonancias Magnéticas (MRI)

Este cuaderno te guiará paso a paso para entrenar un modelo de Inteligencia Artificial
a partir de resonancias magnéticas (MRI) y sus regiones de interés (ROIs).

⚠️ **No necesitas conocimientos de programación**.  
Solo sigue las instrucciones y ejecuta las celdas en orden.

---

## 📁 Organización de los datos (MUY IMPORTANTE)

Antes de empezar, debes tener **una única carpeta dentro del mismo directorio MY DRIVE** que contenga:

- Entre **10 y 50 resonancias magnéticas** en formato **NIfTI** (`.nii` o `.nii.gz`)
- Para **cada resonancia**, un archivo **ZIP** con sus ROIs correspondientes

### 📌 Reglas importantes

- Cada resonancia debe tener **su propio archivo ZIP de ROIs**
- El nombre debe coincidir exactamente:
  - `paciente_01.nii.gz`
  - `paciente_01_roi.zip`
- Dentro de cada archivo ZIP:
  - Hay **un archivo ROI por cada corte de la resonancia**
  - Cada ROI mantiene el **nombre del corte** al que se refiere
- **No cambies los nombres** de los archivos ROI internos


### 📂 Estructura esperada de la carpeta

```
carpeta_datos/
│
├── paciente_01.nii.gz
├── paciente_01.zip
│
├── paciente_02.nii.gz
├── paciente_02.zip
│
├── paciente_03.nii
├── paciente_03.zip
│
└── ...
```


Si la estructura no es correcta, el proceso se detendrá automáticamente.

Cuando tengas la carpeta preparada, continúa con el siguiente paso.


---

## Instalación de librerías necesarias

 **Este paso solo debe ejecutarse una vez**  
Si ya lo ejecutaste anteriormente, puedes saltarlo.


In [ ]:
#@title 🔧 Instalación de Dependencias

# Ejecutar SOLO UNA VEZ

!pip install roifile pillow
!pip install tensorflow nibabel opencv-python numpy

## Paso 1 · Generación de dataset

### Paso 1.1 · Selección de la carpeta de datos
Pulsa el botón y selecciona la carpeta donde tienes **todas las resonancias y sus ROIs**.

El sistema comprobará automáticamente:
- Que la carpeta existe
- Que hay entre 10 y 50 resonancias
- Que cada resonancia tiene su ZIP de ROIs asociado

No necesitas hacer nada más.


In [ ]:
#@title 📁 Elige la carpeta con resonancias

import os
from rich import print
from rich.panel import Panel
from rich.table import Table
from google.colab import drive

# -----------------------------
# MONTAJE DE DRIVE
# -----------------------------
print(Panel("Montando Google Drive...", style="yellow"))
drive.mount('/content/drive')

# -----------------------------
# INSTRUCCIONES
# -----------------------------
print(Panel(
    "PASO 1 · Selección de carpeta de datos\n"
    "Introduce la ruta de la carpeta dentro de tu Google Drive que contiene:\n"
    "- Entre 10 y 50 resonancias (.nii o .nii.gz)\n"
    "- Un archivo ZIP de ROI por cada resonancia\n\n"
    "Ejemplo: /content/drive/MyDrive/carpeta_datos",
    title="🧠 Instrucciones"
))

# El usuario pega la ruta aquí
folder_datos = input("📁 Ruta de la carpeta en Drive: ").strip()

# -----------------------------
# VALIDACIONES
# -----------------------------
def validar_carpeta(folder_datos):
    if not os.path.isdir(folder_datos):
        raise ValueError("❌ La ruta indicada NO es una carpeta válida.\n"
                         "Verifica que la carpeta exista en tu Google Drive y que la ruta sea correcta.")

    files = os.listdir(folder_datos)

    nifti_files = [f for f in files if f.lower().endswith((".nii", ".nii.gz"))]
    zip_files = [f for f in files if f.lower().endswith(".zip")]

    if not (7 <= len(nifti_files) <= 50):
        raise ValueError(f"❌ Se encontraron {len(nifti_files)} resonancias.\n"
                         "Debe haber entre 10 y 50 archivos .nii o .nii.gz.")

    errores = []
    asociaciones = {}

    for nifti in nifti_files:
        base = nifti.replace(".nii.gz", "").replace(".nii", "")
        candidatos = [z for z in zip_files if z.startswith(base)]

        if len(candidatos) == 0:
            errores.append((nifti, "❌ No se encontró ZIP asociado"))
        elif len(candidatos) > 1:
            errores.append((nifti, f"❌ Múltiples ZIPs: {candidatos}"))
        else:
            asociaciones[nifti] = candidatos[0]

    return nifti_files, asociaciones, errores

# -----------------------------
# EJECUCIÓN CON VISUALIZACIÓN
# -----------------------------
try:
    nifti_files, asociaciones, errores = validar_carpeta(folder_datos)

    if errores:
        table = Table(title="Errores detectados en los archivos", show_lines=True)
        table.add_column("Resonancia")
        table.add_column("Problema", style="bold red")
        for nifti, problema in errores:
            table.add_row(nifti, problema)
        print(table)
        print(Panel("❌ Corrige los problemas indicados y vuelve a ejecutar.", style="red"))
    else:
        table = Table(title="✅ Validación exitosa", show_lines=True)
        table.add_column("Resonancia", style="green")
        table.add_column("ZIP asociado", style="green")
        for nifti, zip_file in asociaciones.items():
            table.add_row(nifti, zip_file)
        print(table)
        print(Panel(f"✔ {len(nifti_files)} resonancias detectadas y correctamente asociadas a sus ZIPs.\n"
                    "Puedes continuar al siguiente paso.", style="green"))

except ValueError as e:
    print(Panel(str(e), title="❌ ERROR DE VALIDACIÓN", style="red"))
except Exception as e:
    print(Panel(str(e), title="❌ ERROR INESPERADO", style="red"))


### Paso 1.2 · Conversión de datos

En este paso el sistema convertirá automáticamente las ROIs en **imágenes binarias**.
Se creará una carpeta nueva en la misma ruta donde tenías tus resonancias. **No modificar esta nueva carpeta**


In [ ]:
#@title 💽 Normalización de datos

# =========================================================
# PASO 2 · NORMALIZACIÓN MRI + CONVERSIÓN DE ROIs A BINARIO
# =========================================================

import os
import zipfile
from roifile import ImagejRoi
from PIL import Image, ImageDraw
import tempfile
import nibabel as nib
import numpy as np
from skimage.transform import resize
from rich import print
from rich.panel import Panel
from rich.table import Table

# -----------------------------
# PARÁMETROS DE RESOLUCIÓN
# -----------------------------
TARGET_WIDTH = 120
TARGET_HEIGHT = 120

def convertir_rois_a_binario(folder_datos):
    """
    Convierte los ROIs (ImageJ .roi dentro de ZIP) a imágenes binarias PNG.
    Redimensiona las MRI a TARGET_WIDTH x TARGET_HEIGHT.
    Genera una nueva carpeta <carpeta_original>_procesado.
    """

    base_folder_name = os.path.basename(os.path.normpath(folder_datos))
    output_root = os.path.join(
        os.path.dirname(folder_datos),
        f"{base_folder_name}_procesado"
    )
    os.makedirs(output_root, exist_ok=True)
    print(Panel(f"Los datos procesados se guardarán en:\n[bold green]{output_root}[/bold green]",
                title="📂 Carpeta de salida"))

    nifti_files = [f for f in os.listdir(folder_datos) if f.lower().endswith((".nii", ".nii.gz"))]

    if len(nifti_files) == 0:
        print(Panel("❌ No se encontraron resonancias en la carpeta indicada", style="red"))
        return

    print(Panel("El proceso puede tardar unos minutos...", style="yellow"))

    resumen = []

    for nifti in nifti_files:
        base = nifti.replace(".nii.gz", "").replace(".nii", "")
        zip_name = f"{base}.zip"

        nifti_path = os.path.join(folder_datos, nifti)
        zip_path = os.path.join(folder_datos, zip_name)

        paciente_dir = os.path.join(output_root, base)
        rois_dir = os.path.join(paciente_dir, "ROIS")
        os.makedirs(rois_dir, exist_ok=True)

        try:
            if not os.path.exists(zip_path):
                resumen.append((base, "❌ ZIP de ROIs no encontrado, paciente omitido"))
                continue

            # -----------------------------
            # REDIMENSIONAR MRI
            # -----------------------------
            try:
                nii_img = nib.load(nifti_path)
                data = nii_img.get_fdata()
            except Exception as e:
                resumen.append((base, f"❌ Error al cargar MRI: {e}"))
                continue

            slices_resized = []
            for i in range(data.shape[2]):
                slice_img = data[:, :, i]
                slice_resized = resize(slice_img, (TARGET_HEIGHT, TARGET_WIDTH),
                                       preserve_range=True, anti_aliasing=True)
                slices_resized.append(slice_resized)

            data_resized = np.stack(slices_resized, axis=2)
            new_nii = nib.Nifti1Image(data_resized, affine=nii_img.affine)
            out_nii_path = os.path.join(paciente_dir, nifti)
            nib.save(new_nii, out_nii_path)

            # -----------------------------
            # PROCESAR ROIs
            # -----------------------------
            with tempfile.TemporaryDirectory() as tmpdir:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(tmpdir)

                    roi_files = [f for f in zip_ref.namelist() if f.lower().endswith(".roi")]
                    if not roi_files:
                        resumen.append((base, "❌ No se encontraron ROIs dentro del ZIP"))
                        continue

                    for roi_file in roi_files:
                        roi_path = os.path.join(tmpdir, roi_file)
                        try:
                            roi = ImagejRoi.fromfile(roi_path)
                            coords = roi.coordinates()
                        except Exception as e:
                            resumen.append((base, f"❌ Error al leer ROI {roi_file}: {e}"))
                            continue

                        roi_scaled = [
                            (
                                float(x) * TARGET_WIDTH / data.shape[0],
                                float(y) * TARGET_HEIGHT / data.shape[1]
                            )
                            for x, y in coords
                        ]

                        mask = Image.new("L", (TARGET_WIDTH, TARGET_HEIGHT), 0)
                        draw = ImageDraw.Draw(mask)
                        draw.polygon(roi_scaled, outline=255, fill=255)

                        out_name = os.path.basename(roi_file).replace(".roi", ".png")
                        mask.save(os.path.join(rois_dir, out_name))

            resumen.append((base, "✅ Procesado correctamente"))

        except Exception as e:
            resumen.append((base, f"❌ Error inesperado: {e}"))

    # -----------------------------
    # MOSTRAR RESUMEN
    # -----------------------------
    table = Table(title="Resumen del procesamiento", show_lines=True)
    table.add_column("Paciente")
    table.add_column("Estado")

    for paciente, estado in resumen:
        if "✅" in estado:
            table.add_row(paciente, f"[green]{estado}[/green]")
        else:
            table.add_row(paciente, f"[red]{estado}[/red]")

    print(table)
    print(Panel("✅ Proceso completado. Revisa los pacientes con errores y corrige antes de continuar.",
                style="blue"))

    return output_root
# -----------------------------
# EJECUCIÓN DEL PASO
# -----------------------------
try:
    output_root = convertir_rois_a_binario(folder_datos)
except Exception as e:
    print(Panel(str(e), title="❌ ERROR INESPERADO", style="red"))


## Paso 2 · Entrenamiento del modelo

### Paso 2.1 · Comprobación de alineamiento

En algunos casos, la orientación de la MRI puede no coincidir con la de los cortes convertidos en el paso anterior.
Por ello, asegúrate de comprobar, en este paso, que la alineación de la superposición de cada corte es correcta:

In [ ]:
#@title ✔️ Comprobación de Alineamiento

# =============================================================
# PASO 3 · COMPROBACIÓN DE ALINEACIÓN MRI ↔ ROI
# =============================================================

import os
import numpy as np
import nibabel as nib
from PIL import Image
import matplotlib.pyplot as plt

DATASET_PATH = output_root
IMG_TARGET = 120

# -----------------------------
# Funciones de carga
# -----------------------------
def load_nifti_slices(nifti_path):
    """Carga un NIfTI y devuelve un volumen normalizado."""
    nii = nib.load(nifti_path)
    vol = nii.get_fdata()

    # Normalización 0–1
    vol = (vol - np.min(vol)) / (np.max(vol) - np.min(vol))

    # Rotación según tu pipeline
    vol = np.rot90(vol, k=3, axes=(0, 1))
    vol = np.flip(vol, axis=1)

    return vol

def load_mask(mask_path):
    """Lee una máscara PNG blanco y negro → binaria (0/1)."""
    img = Image.open(mask_path).convert("L")
    mask = np.array(img, dtype=np.uint8)
    mask = (mask > 127).astype(np.uint8)
    return mask

# -----------------------------
# Preparar dataset
# -----------------------------
def get_training_pairs():
    X, Y = [], []

    for case in sorted(os.listdir(DATASET_PATH)):
        case_path = os.path.join(DATASET_PATH, case)

        # MRI de paciente
        nifti_file = None
        for f in os.listdir(case_path):
            if f.lower().endswith((".nii", ".nii.gz")):
                nifti_file = os.path.join(case_path, f)
                break
        if nifti_file is None:
            print(f"❌ No se encontró NIfTI para {case}, se salta")
            continue

        # Carpeta ROIS
        mask_folder = os.path.join(case_path, "ROIS")
        if not os.path.exists(mask_folder):
            print(f"❌ No se encontró carpeta ROIS para {case}, se salta")
            continue

        vol = load_nifti_slices(nifti_file)

        for fname in sorted(os.listdir(mask_folder)):
            if not fname.endswith(".png"):
                continue

            slice_idx = int(fname.split("-")[0]) - 1

            if slice_idx < 0 or slice_idx >= vol.shape[2]:
                continue

            img = vol[:, :, slice_idx]
            mask = load_mask(os.path.join(mask_folder, fname))

            # Eliminar ejes extra
            img_s = np.squeeze(img)
            mask_s = np.squeeze(mask)

            # Convertimos a PIL
            img_p = Image.fromarray((img_s*255).astype(np.uint8))
            mask_p = Image.fromarray(mask_s.astype(np.uint8))

            # Solo redimensionar si no son 120x120
            if img_p.size != (IMG_TARGET, IMG_TARGET):
                img_p = img_p.resize((IMG_TARGET, IMG_TARGET))
            if mask_p.size != (IMG_TARGET, IMG_TARGET):
                mask_p = mask_p.resize((IMG_TARGET, IMG_TARGET), resample=Image.NEAREST)

            # Convertimos a array y añadimos canal
            img_p = np.array(img_p, dtype=np.float32)/255.0
            img_p = img_p[..., np.newaxis]

            mask_p = np.array(mask_p, dtype=np.float32)[..., np.newaxis]

            X.append(img_p)
            Y.append(mask_p)

    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.float32)

    print("✔ Dataset preparado")
    print("   Imágenes:", X.shape)
    print("   Máscaras:", Y.shape)

    return X, Y

X, Y = get_training_pairs()
print("📊 Dataset cargado")

# -----------------------------
# Visualización de overlay
# -----------------------------
def visualize_overlay(X, Y, idx):
    img = X[idx, :, :, 0]
    mask = Y[idx, :, :, 0]

    plt.figure(figsize=(12,4))

    # Imagen
    plt.subplot(1,3,1)
    plt.imshow(img, cmap="gray")
    plt.title("Imagen (input CNN)")
    plt.axis("off")

    # Máscara
    plt.subplot(1,3,2)
    plt.imshow(mask, cmap="gray")
    plt.title("Máscara (label)")
    plt.axis("off")

    # Superposición
    plt.subplot(1,3,3)
    plt.imshow(img, cmap="gray")
    plt.imshow(mask, cmap="Greens", alpha=0.4)
    plt.title("Superposición")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# Visualizamos 5 ejemplos aleatorios
for idx in np.random.choice(len(X), size=5, replace=False):
    visualize_overlay(X, Y, idx)


### Paso 2.2 · ¡Listos para entrenar!

Todo el dataset ya está preparado y listo para el entrenamiento.

Al ejecutar la siguiente celda:

- Se generará **el modelo de red neuronal CNN**.
- El proceso puede tardar un tiempo, dependiendo de:
  - Número de muestras en tu dataset.
  - Capacidad de computación de tu equipo o Google Colab.

Durante el entrenamiento podrás:

- Observar las **fases de aprendizaje**.
- Ver cómo el modelo hace **predicciones sobre las imágenes**.

⚠️ Deja que la celda termine de ejecutarse sin interrumpirla para obtener resultados óptimos.


In [ ]:
#@title 🤺 Entrenamiento del Modelo

# =========================================================
# U-NET COMPLETO PARA COLAB
# =========================================================

import tensorflow as tf
from tensorflow.keras import models
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D, Input, concatenate
from tensorflow.keras.callbacks import Callback
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Parámetros
# -----------------------------
IMG_TARGET = 120  # Tamaño de las MRI y ROIs procesadas
BATCH_SIZE = 4

# -----------------------------
# Funciones de pérdida y métricas
# -----------------------------
def dice_coef(y_true, y_pred, smooth=1):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

# -----------------------------
# Callback para visualizar predicciones en tiempo real
# -----------------------------
class DisplayPredictions(Callback):
    def __init__(self, X_sample, Y_sample):
        self.X_sample = X_sample
        self.Y_sample = Y_sample

    def on_epoch_end(self, epoch, logs=None):
        idx = np.random.randint(len(self.X_sample))
        x = self.X_sample[idx:idx+1]  # batch 1
        y_true = self.Y_sample[idx]

        y_pred = self.model.predict(x)[0,:,:,0]

        plt.close('all')  # cerrar figuras previas
        plt.figure(figsize=(12,4))

        plt.subplot(1,3,1)
        plt.imshow(x[0,:,:,0], cmap='gray')
        plt.title("Input")

        plt.subplot(1,3,2)
        plt.imshow(y_true[:,:,0], cmap='Greens')
        plt.title("Mask")

        plt.subplot(1,3,3)
        plt.imshow(x[0,:,:,0], cmap='gray')
        plt.imshow(y_pred, cmap='Reds', alpha=0.4)
        plt.title(f"Predicción Epoch {epoch+1}")

        plt.show()

# -----------------------------
# Bloque conv para U-Net
# -----------------------------
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x

# -----------------------------
# Modelo U-Net
# -----------------------------
def unet_model():
    inputs = Input((IMG_TARGET, IMG_TARGET, 1))  # MRI de tamaño 120x120

    c1 = conv_block(inputs, 32)
    p1 = MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = MaxPooling2D()(c3)

    bn = conv_block(p3, 256)

    u3 = UpSampling2D()(bn)
    u3 = concatenate([u3, c3])
    c4 = conv_block(u3, 128)

    u2 = UpSampling2D()(c4)
    u2 = concatenate([u2, c2])
    c5 = conv_block(u2, 64)

    u1 = UpSampling2D()(c5)
    u1 = concatenate([u1, c1])
    c6 = conv_block(u1, 32)

    outputs = Conv2D(1, 1, activation="sigmoid")(c6)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=bce_dice_loss,
        metrics=[dice_coef]
    )
    return model

# -----------------------------
# EJEMPLO DE ENTRENAMIENTO CON UN SUBSET
# -----------------------------

# Test de overfitting con un pequeño subset
model = unet_model()
X_test = X[:28]  # 28 slices
Y_test = Y[:28]

callback_vis = DisplayPredictions(X_test, Y_test)

history_overfit = model.fit(
    X_test, Y_test,
    batch_size=4,
    epochs=100,
    validation_split=0.1,
    callbacks=[callback_vis]
)

# -----------------------------
# Entrenamiento completo
# -----------------------------
model_full = unet_model()  # reiniciar pesos
callback_vis_full = DisplayPredictions(X[:5], Y[:5])  # visualización con subset pequeño

history = model_full.fit(
    X, Y,
    batch_size=BATCH_SIZE,
    epochs=35,
    validation_split=0.1,
    callbacks=[callback_vis_full]
)

print("🎉 Entrenamiento terminado")

# -----------------------------
# Visualización final de métricas
# -----------------------------
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='loss (train)')
plt.plot(history.history['val_loss'], label='loss (val)')
plt.title("Pérdida durante el entrenamiento")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['dice_coef'], label='dice_coef (train)')
plt.plot(history.history['val_dice_coef'], label='dice_coef (val)')
plt.title("Dice coefficient durante el entrenamiento")
plt.xlabel("Época")
plt.ylabel("Dice")
plt.legend()

plt.show()


### Paso 2.3 · Guardar el modelo

Descarga el modelo entrenado para utilizarlo.

In [ ]:
#@title 🦾 Guardar el Modelo

import os
from google.colab import drive
from rich import print
from rich.panel import Panel


# -----------------------------
# Selección de carpeta de guardado
# -----------------------------
save_dir = input("📁 Introduce la ruta dentro de tu Google Drive donde quieres guardar el modelo:\n"
                 "(Ejemplo: /content/drive/MyDrive/modelos): ").strip()

# Crear la carpeta si no existe
os.makedirs(save_dir, exist_ok=True)

# -----------------------------
# Guardar modelo
# -----------------------------
model_filename = "segmentation_model.h5"
model_path = os.path.join(save_dir, model_filename)

try:
    model.save(model_path)
    print(Panel(f"💾 Modelo guardado correctamente en:\n[bold green]{model_path}[/bold green]", style="green"))
except Exception as e:
    print(Panel(f"❌ Error al guardar el modelo:\n{e}", style="red"))
